In [ ]:
import speech_recognition as sr
import pyttsx3
import pyjokes
import pywhatkit
import boto3

In [ ]:
Access_key="Enter your Access key here"
Secret_key="Enter your Secret key here"

In [ ]:
Session=boto3.Session(aws_access_key_id=Access_key,aws_secret_access_key=Secret_key,region_name="ap-south-1")
s3=boto3.client("s3",aws_access_key_id=Access_key,aws_secret_access_key=Secret_key)
ec2=boto3.resource("ec2",aws_access_key_id=Access_key,aws_secret_access_key=Secret_key,region_name="ap-south-1")
ec2_client = boto3.client("ec2", aws_access_key_id=Access_key, aws_secret_access_key=Secret_key, region_name="ap-south-1")
    
def list():
    response=s3.list_buckets()
    bucket_name=[bucket['Name']
                 
    for bucket in response["Buckets"]]
    print("Buckets in system :",bucket_name)
def get_infrastructure_counts():
    
    s3_response = s3.list_buckets()
    bucket_count = len(s3_response.get("Buckets", []))
    instance_count = len([instance for instance in ec2.instances.all()])
    return f"There are currently {bucket_count} number of buckets and {instance_count} number of instances in your system"
  
     

In [ ]:
start=pyttsx3.init()
recognizer = sr.Recognizer()
print("System is initializing...")
print("""
            1. Create Bucket
            2. Upload File
            3. Download File
            4. Delete Object
            5. Delete Bucket
            6. Status/Infrastructure
            7. Deploy/Launch
            8. Crack a Joke
            9. Stop Instance
            10.List down Bucket 
            11.Terminate Instance
            12.Tutorial
            13.Documentation
            14.list down Instance
            15.Stop
            """)
start.say("Cloud Monitoring Terminal Initialized. Ready for voice verification.")
start.runAndWait()
my_name= None
while True:
    
    with sr.Microphone() as source:
        print("\n[LISTENING] State your command...")
        
        
        recognizer.adjust_for_ambient_noise(source, duration=0.5)
        
        try:
            
            audio_data = recognizer.listen(source)
            
            
            query = recognizer.recognize_google(audio_data).lower()
            
            print(f"-> Text recognized: '{query}'")
            
            
            if "status" in query or "infrastructure" in query:
                print("[EXECUTE] Fetching AWS infrastructure state via Boto3...")
                status_report=get_infrastructure_counts()
                print(status_report)
                start.say(status_report)
                start.runAndWait()
                
            elif "deploy" in query or "launch" in query:
                print("[EXECUTE] Deploying a t3.micro instance...")
                imageid=str(input("Enter image id u want"))
                ec2.create_instances(ImageId=imageid,MaxCount=1,MinCount=1,InstanceType="t3.micro")
                print("Instance Launched!!")
                start.say("instance launched")
                start.runAndWait()
                
            elif "create bucket" in query :
                my_name=input("Enter the name of the bucket u want to make:")
                print("[EXECUTE]Creating a bucket with name :",my_name)
                s3.create_bucket(Bucket=my_name,CreateBucketConfiguration={
                'LocationConstraint':'ap-south-1'
                })
                print("Bucket Created!!")
                start.say("bucket created")
                start.runAndWait()
                   
            elif "crack a joke" in query :
                print("[EXECUTE] Accessing humor protocols...")
                joke=pyjokes.get_joke()
                engine=pyttsx3.init()
                engine.say(joke)
                print(joke)
                engine.runAndWait()
            
                
            elif "upload file" in query :
                if not my_name:
                    my_name = input("No active bucket set. Enter target bucket name: ")
                key = input("Enter the name of the file to store in the bucket: ")
                file = input("Enter the name of the file to upload in the bucket: ")
                print("Command detected. Uploading the file...")
                s3.upload_file(file, Bucket=my_name, Key=key)
                print("File Uploaded!!")
                start.say("File uploaded")
                start.runAndWait()
                
            elif "download file" in query :
                if not my_name:
                    my_name = input("No active bucket set. Enter target bucket name: ")
                Filename = input("Enter the name of the file to store in the local computer: ")
                key = input("Enter the name of the file to download in the local computer: ")
                print("Command detected. Downloading the file...")
                s3.download_file(Bucket=my_name, Filename=Filename, Key=key)
                print("File Downloaded!!")
                start.say("File downloaded")
                start.runAndWait()
                
            elif "delete bucket" in query :
                bucketname=input("enter the bucket name u want to delete")
                print("command detected.deleting the bucket...")
                s3.delete_bucket(Bucket=bucketname)
                print("Bucket Deleted!!")
                start.say("bucket deleted")
                start.runAndWait()
                
            elif "delete object" in query :
                if not my_name:
                    my_name = input("No active bucket set. Enter target bucket name: ")
                filename = input("Enter the filename u want to delete from the bucket: ")
                print("Command detected. Deleting the object...")
                s3.delete_object(Bucket=my_name, Key=filename)
                print("Object Deleted!!")
                start.say("object deleted")
                start.runAndWait()
                
            elif "stop instance" in query :
                instance_id=input("Enter instance Id: ")
                ec2_client.stop_instances( InstanceIds=[instance_id])
                print("[SYSTEM] stopping the instance.")
                print("Instance Stopped!!")
                start.say("instance stopped")
                start.runAndWait() 
                
            elif "list down bucket" in query :
                print("command detected.listing down the buckets...")
                list()
                 
            elif "list down instance" in query :
                print("command detected.listing down the instances...")
                for instance in ec2.instances.all():
                 print(f"ID: {instance.id} | State: {instance.state['Name']}")
                     
            elif "terminate instance" in query :
                instance_id=input("Enter instance Id: ")
                ec2_client.terminate_instances(InstanceIds=[instance_id])
                print("[SYSTEM] terminating  the instance.")

            elif "tutorial" in query : 
                print("command detected.playing tutorials on youtube")
                search="https://www.youtube.com/watch?v=Ld1_vb1fH6c&list=PLjl2dJMjkDjlcI3SQErSq4UMX3okzafvO"
                pywhatkit.playonyt(search)
                
            elif "documentation" in query : 
                print("command detected.opening the Documentation of python boto3")
                pywhatkit.search("https://docs.aws.amazon.com/boto3/latest/")   
                
            elif "stop" in query : 
                print("command detected.Shutting down the console.goodbye")
                break
            else:
                print("[ERROR] Command not recognized. Please repeat.")
                
        except sr.UnknownValueError:
            print("[SYSTEM] Audio unclear. Monitoring baseline frequency...")
            
        except sr.RequestError:
            print("[SYSTEM] Connection error. Could not reach speech API.")